#Data Keuangan



##Sumber Data
Dataset dari kaggle (personal-finance-tracker-dataset) : https://www.kaggle.com/datasets/khushikyad001/personal-finance-tracker-dataset?resource=download
<br>



#import Libarary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

: 

#LOAD DATA

In [ ]:
data_df = pd.read_csv('raw data keuangan.csv')
data_df.head()

Okey dari dataset personal finance tracker ini, tidak semua kolom akan dipakai dalam data financial treacker feature kami. Kami akan melakukan Feature Selection untuk memiilih kolom kolom yang akan di pakai pada fitur kami yaitu mengenai pemasukan, pengeluaran, dana darurat, rasion hutang/cicilan perbulan

#ASSESSING DATA

In [ ]:
data_df.info()

Tidak ada data yang null, dan hanya fitur 'date' yang perlu disesuaikan tipe datanya (date), dan 'user' id yang disesuaikan tipe (str)

In [ ]:
data_df.isna().sum()

check data yang null, ternyata tidak ada yang null

In [ ]:
data_df.duplicated().sum()

Tidak ada data yang duplicated

In [ ]:
data_df.describe()

#DATA CLEANING & REPROCESSING

In [ ]:
df_clean_data = data_df.copy()

Mengcopy df_featur_kami untuk dataframe df_clean_data

In [ ]:
df_clean_data['date'] = pd.to_datetime(df_clean_data['date'])

Mengsetting tipe data fitur 'date' menjadi (date)

In [ ]:
df_clean_data['user_id'] = df_clean_data['user_id'].astype(str)

Dan juga mensetting tipe data 'user_id' menjadi str

In [ ]:
df_clean_data.info()

Okey, data sudah bersih

#EDA

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_clean_data['monthly_income'], color='skyblue')
plt.title('Deteksi Outlier pada Pemasukan Bulanan', fontweight='bold')
plt.xlabel('Monthly Income')
plt.show()

okey ada outlier pada pendapatan perbulan namun kami tidak melakukan ddrop ataupun clipping karena :
- Outlier Atas (Pemasukan Sangat Tinggi): Titik-titik anomali di sisi kanan merepresentasikan High Net-Worth Individuals (pengguna bermodal besar). Data ini sangat bernilai tinggi (high information value) karena algoritma K-Means akan menggunakan angka ekstrem ini untuk menarik mereka ke dalam cluster Siap Investasi (Agresif).
- Outlier Bawah (Pemasukan Sangat Rendah): Titik-titik di sisi kiri merepresentasikan pengguna dengan kerentanan finansial tertinggi. Mempertahankan nilai asli mereka sangat krusial agar model dapat dengan tegas mengenali dan memisahkan mereka ke dalam cluster Belum Siap Investasi.

In [ ]:
df_clean_data.describe()

Terdapat beberapa temuan pada hasil describe() :

- Kelengkapan Data & Rentang Waktu, nilai count yang seragam di angka 3.000 menunjukkan bahwa tidak ada missing values (data kosong) pada seluruh kolom numerik maupun tanggal.

- Rata-rata pengeluaran bulanan (monthly_expense_total) pengguna adalah 3.011. Namun, nilai maksimal dari dana darurat (emergency_fund) di seluruh dataset ini hanyalah 2.585.

- Rasio utang terhadap pemasukan (debt_to_income_ratio) rata-rata berada di angka 0.35 (35%). Sementara itu, rata-rata nominal investasi (investment_amount) berada di angka 400, dengan variasi (std) yang tidak terlalu ekstrem, menunjukkan adanya minat investasi dasar dari pengguna.

In [ ]:
userboros = df_clean_data[df_clean_data['monthly_expense_total'] > df_clean_data['monthly_income']]

print(f"Jumlah user dengan pengeluaran > pemasukan: {len(userboros)} user")
display(userboros[['user_id', 'monthly_income', 'monthly_expense_total']].head(10))

Setelaah dicoba ditelaah ternyata ada 651 user yang memimiliki pengeluaran perbulannya lebih banyak dari pada income perbulannnya.

In [ ]:
plt.figure(figsize=(12, 8))

# kolom numerik dataset
numeric_cols = df_clean_data.select_dtypes(include=['int64', 'float64']).columns

# heatmap dari seluruh fitur numerik
sns.heatmap(df_clean_data[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Peta Korelasi Keseluruhan Fitur Sebelum Seleksi', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

dari *Heatmap* Korelasi terhadap seluruh fitur mentah, terdapat *insight* penting yang mendasari dalam tahap *Feature Selection*:

1. Kebebasan dari Multikolinearitas: Lima fitur utama yang akan dipilih (`monthly_income`, `monthly_expense_total', `emergency_fund`, `investment_amount', dan `debt_to_income_ratio`) saling menunjukkan angka korelasi yang sangat mendekati 0 (berkisar antara -0.05 hingga 0.02). Hal ini sangat ideal untuk algoritma *clustering* K-Means, karena setiap fitur independen tanpa adanya pembobotan ganda.
2. Pada sudut kanan bawah matriks, terlihat bahwa fitur `actual_savings` memiliki korelasi yang sangat kuat dengan `monthly_income` (0.74) dan `monthly_expense_total` (-0.60). tapi memasukkan `actual_savings` ke dalam dataset pemodelan bersamaan dengan fitur pemasukan dan pengeluaran akan membuat model K-Means menjadi bias terhadap satu aspek finansial saja.

Berdasarkan visualisasi Heatmap Korelasi, kita melakukan seleksi fitur sebelum masuk ke tahap pemodelan K-Means. Pemilihan 5 fitur numerik utama dan 2 fitur identitas ini didasari oleh alasan analitis berikut:

- Independensi Fitur : Kelima fitur finansial yang dipilih (monthly_income, monthly_expense_total, emergency_fund, investment_amount, dan debt_to_income_ratio) memiliki nilai korelasi antar-variabel yang sangat rendah. Hal ini sangat ideal untuk algoritma K-Means, karena membuktikan bahwa kelima metrik tersebut saling independen, tidak ada informasi yang tumpang tindih, dan akan mencegah model memberikan bobot ganda (double counting) pada satu aspek finansial tertentu.

- Menghindari Jebakan Redundansi, Kita secara sengaja mengeleminasi fitur-fitur yang memiliki korelasi tinggi. Sebagai contoh, fitur actual_savings tidak dimasukkan karena memiliki korelasi yang kuat dengan monthly_income (0.74) dan monthly_expense_total (-0.60). Memasukkan fitur tersebut hanya akan memberikan informasi redundan yang berpotensi membiaskan hasil clustering.

- Kolom date dan user_id tetap diikutsertakan dalam df_feature_kami murni sebagai metadata pengenal. Kolom ini tidak akan dimasukkan ke dalam perhitungan matematis algoritma (akan diset sebagai index), melainkan digunakan untuk melacak dan memetakan hasil cluster kembali kepada masing-masing pengguna di tahap akhir analisis.

#Feature Selection

##Kolom yang akan dipakai:
- 'Date'
- 'user_id'
- 'monthly_income'
- 'monthly_expense_total'
- 'emergency_fund',
- 'investment_amount',     
- 'debt_to_income_ratio'

In [ ]:
feature_kami = [
    'date',
    'user_id',
    'monthly_income',
    'monthly_expense_total',
    'emergency_fund',
    'investment_amount',
    'debt_to_income_ratio'
]

df_feature_kami = df_clean_data[feature_kami].copy()

Memilih kolom yang akan dipakai, dan mengcopy data_df untuk dataframe baru (df_feature_kami)

In [ ]:
df_feature_kami.head()

#FEATURE ENGINEERING

In [ ]:
df_keuangan = df_feature_kami.copy()

Mencopy dataframe df_clean_data untuk dataframe baru df_keuangan

In [ ]:
np.random.seed(42)
df_keuangan['age'] = np.random.randint(18, 31, size=len(df_keuangan))

Membuat fitur baru 'age' di mana mengkunci random terlebih dahulu agar setiap di run tidak perlu mengack kembali, dan range pengambilan value acak nya rentan 18-31

##Menkonversi value fitur 'monthly_income', 'monthly_expense_total' 'investment_amount', 'emergency_fund' dalam rupiah (IDR)

In [ ]:
KURS_USD_TO_IDR = 17478

Menetapkan variabel kurs USD to Rupiah (IDR)

In [ ]:
kolom_uang = [
    'monthly_income',
    'monthly_expense_total',
    'investment_amount',
    'emergency_fund',
]

Memilih fitur yang akan dikonversi

In [ ]:
for col in kolom_uang:
    if col in df_keuangan.columns:
        df_keuangan[col] = df_keuangan[col] * KURS_USD_TO_IDR

Melakukan pengulangan pada kolom uang untuk mengalikan value pada masing masing fitur dengan KURS_USD_TO_IDR

In [ ]:
df_keuangan.head(21)

In [ ]:
df_keuangan.to_csv('data keuangan_clean.csv')

Okey, dataset keuangan clean dan telah malakukan fatur engineerung done

#CLUSTERING (karakteristik Investor)

Membuat cluster/kategori masing masing user apakah sudah siap untuk beriventasi? dengan algoritma machine learning Unsupervised Learning : K-Means.

In [ ]:
kolom_cluster = ['investment_amount', 'debt_to_income_ratio', 'score_emergency']

Memilih 3 fitur : 'investment_amount', 'debt_to_income_ratio', 'score_emergency'. sebagai 3 pilar profil resiko user untuk algoritma kmeans mebagi cluster

In [ ]:
# kalkulasi 'Rasio Dana Darurat'
df_keuangan['Rasio Dana Darurat'] = (df_keuangan['emergency_fund'] / df_keuangan['monthly_expense_total']).round(2)

# score minimal
df_keuangan['score_emergency'] = np.where(
    df_keuangan['Rasio Dana Darurat'] >= 6,
    40,
    (df_keuangan['Rasio Dana Darurat'] / 6) * 40
).round(2)

# outlier handling
for col in kolom_cluster:
    upper_limit = df_keuangan[col].quantile(0.95) # Ambil batas 95%
    df_keuangan[col] = df_keuangan[col].clip(upper=upper_limit)

- Mengkalkulasi 'Rasio Dana Darurat', di mana fitur 'emergency_fund' dibagi fitur 'monthly_experience_total'.

- score minimal, memiliki dana darurat 6 bulan agar termasuk kategori "Sangat Aman", Jika ada user yang punya dana darurat untuk lebih 6 bualan mewndapat score aman yaitu 40.
- mencari nilai batas atas pada persentil ke-95 (quantile(0.95)). titik di mana 5% orang paling ekstrem/kaya berada, Fungsi .clip(upper=upper_limit) akan memangkas nilai orang-orang super kaya tersebut agar sama dengan nilai batas 95% yang didapat.

In [ ]:
x = df_keuangan[kolom_cluster]

Membuat varibel x yang berisi 3 fitur inti

##Scaling

In [ ]:
scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(x)

melakukan normalisasi data (scaling) dengan menginisialisasi fungsi MinMaxScaler() ke dalam variabel scaler. Selanjutnya, dilakukan proses fit_transform pada variabel x untuk mempelajari nilai minimum dan maksimum dari setiap kolom, sekaligus mengubah seluruh angka tersebut ke dalam skala yang seragam, yaitu 0 hingga 1."

##Mencari jumlah cluster menggunakan Elbow Method

In [ ]:
# elbow method
inertia = []
K_range = range(1, 10)

for k in K_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_test.fit(x_scaled)
    inertia.append(kmeans_test.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker='o', linestyle='--', color='b')
plt.title('Elbow Method untuk Mencari Jumlah Cluster Optimal')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Inertia (Jarak Error)')
plt.grid(True)
plt.show()

Terlihat bahwa titik siku (elbow) berada di angka $k=3$. Penurunan nilai Inertia (Jarak Error) sangat tajam dari cluster 1 ke 3, namun mulai melandai secara stagnan pada cluster ke-4 dan seterusnya. Dapat diambil membagi dalam 3 cluster itu pilihan yang pas.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_keuangan['cluster_investor'] = kmeans.fit_predict(x_scaled)

In [ ]:
sil_score = silhouette_score(x_scaled, df_keuangan['cluster_investor'])

In [ ]:
print(f"Silhouette Score model: {sil_score:.3f}")

In [ ]:
profiling_cols = [col for col in kolom_cluster if col != 'user_id']
profil_cluster = df_keuangan.groupby('cluster_investor')[profiling_cols].mean().round(2)
display(profil_cluster)

In [ ]:
kamus_cluster = {
    0: 'Siap Investasi (Agresif)',
    1: 'Belum Siap Investasi',
    2: 'Siap Investasi (Konservatif)'
}

Kamus untuk keterangan setiap label cluster (0,1,2)

In [ ]:
df_keuangan['status_kesiapan_invest'] = df_keuangan['cluster_investor'].map(kamus_cluster)

In [ ]:
df_keuangan.head(339)

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_keuangan,
    x='debt_to_income_ratio',   # Sumbu X: Rasio utang (0 sampai 1)
    y='investment_amount',      # Sumbu Y: Uang yang diinvestasikan
    hue='status_kesiapan_invest',
    palette='viridis',
    s=100,
    alpha=0.8,
    edgecolor='black'
)


plt.title('Persebaran Profil Investor', fontsize=14, fontweight='bold')
plt.xlabel('Debt to Income', fontsize=12)
plt.ylabel('Nominal Investasi (Rupiah)', fontsize=12)

#  garis bantu (grid)
plt.grid(True, linestyle='--', alpha=0.6)

plt.legend(
    title='Kelompok (Cluster)',
    title_fontsize='11',
    loc='upper right',
    bbox_to_anchor=(1, -0.15),
    ncol=3
)

plt.tight_layout()
plt.show()

#MERGE Data Kepemilikan Saham

In [ ]:
df_saham = pd.read_csv('lq45_clean_dataset.csv')

Meload dataset lq45 yang sudah clean untuk digabung pada dataset keuangan ini, sebagai kepemilikan saham masing masing user

In [ ]:
daftar_saham = df_saham['Ticker'].unique()

In [ ]:
jumlah_user = len(df_keuangan)

In [ ]:
np.random.seed(42)

In [ ]:
df_final = df_keuangan.copy()

In [ ]:
df_final['ticker_saham'] = 'Belum Ada Portfolio'
df_final['jumlah_lot'] = 0
df_final['average_price'] = 0

In [ ]:
mask_punya_uang = df_final['investment_amount'] > 0
jumlah_investor = mask_punya_uang.sum()

In [ ]:
df_final.loc[mask_punya_uang, 'ticker_saham'] = np.random.choice(daftar_saham, jumlah_investor)
df_final.loc[mask_punya_uang, 'jumlah_lot'] = np.random.randint(1, 50, jumlah_investor)
df_final.loc[mask_punya_uang, 'average_price'] = np.random.randint(1000, 5000, jumlah_investor)

In [ ]:
kolom_tampilan = ['investment_amount', 'Cluster_AI', 'ticker_saham', 'jumlah_lot']

In [ ]:
df_final.head(200)

In [ ]:
df_final.to_csv('dataset_keuangan.csv', index=False)

Dataset Keuangan yang final telah siap

#Explanatory Analysis

- Dari seluruh user bagaimana proposi dari setiap cluster?
- bagaimana karakterisitik finansial masing masing cluster?
- pakah orang yang gajinya besar (monthly_income) pasti investasinya (investment_amount) besar juga?

In [ ]:
# proporsi kesiapan investasi
plt.figure(figsize=(8, 6))
# jumlah user di tiap cluster
ax = sns.countplot(
    data=df_keuangan,
    x='status_kesiapan_invest',
    palette='Set2',
    order=df_keuangan['status_kesiapan_invest'].value_counts().index
)

total = len(df_keuangan)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x_pos = p.get_x() + p.get_width() / 2
    y_pos = p.get_height() + 5
    ax.annotate(percentage, (x_pos, y_pos), ha='center', fontweight='bold')

plt.title('Distribusi Profil Kesiapan Investasi User', fontsize=14, fontweight='bold')
plt.xlabel('Status Investasi', fontsize=12)
plt.ylabel('Jumlah User', fontsize=12)
plt.tight_layout()
plt.show()

kategori "Belum Siap Investasi" sebanyak (41.8%). Sementara itu, pengguna yang sudah siap berinvestasi terbagi menjadi dua profil risiko, yaitu "Konservatif" (30.3%) dan "Agresif" (27.9%). Meskipun kelompok "Belum Siap" adalah mayoritas tunggal, jika digabungkan, lebih dari separuh user kita (58.2%) sebenarnya sudah memiliki kapasitas untuk berinvestasi.

In [ ]:
# karakteristik tiap cluster
kolom_analisis = ['score_emergency', 'investment_amount', 'debt_to_income_ratio']
judul_analisis = ['Skor Dana Darurat', 'Nominal Investasi', 'Rasio Utang (Debt to Income)']

plt.figure(figsize=(18, 5))

for i, col in enumerate(kolom_analisis):
    plt.subplot(1, 3, i+1)
    sns.boxplot(
        data=df_keuangan,
        x='status_kesiapan_invest',
        y=col,
        palette='Set1'
    )
    plt.title(f'Perbandingan {judul_analisis[i]}', fontweight='bold')
    plt.xlabel('')
    plt.ylabel(judul_analisis[i])
    plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

Algoritma K-Means berhasil memisahkan profil user dalam 3 cluster :
- pertama di grafik perbandingan skor dana darurat, K-Means mendeteksi orang-orang yang rentan untuk berinvestasi dari keminimannya dana darurat artinya sangat tidak aman user tersebut untuk melakukan investasi.
- kedua di grafik perbandingan nominal investasi, median user berada di posisi tertinggi, membuktikan user cluster ini berani menaruh modal paling besar. dan juga badan kotaknya sangat lebar, artinya rentang modal di kelompok ini sangat bervariasi, mulai dari investor kelas menengah hingga investor yang sudah berpengalaman.
- ketiga di grafik perbandingan rasio utang, cluster yang belum siap investasi cenderung memiliki rasio utang yang lebih tinggi, dan cluster investor konservatif dan agresif memilki rasio utang yang rendah, tapi juga kita bisa lihat investor agresif memilki rasio utang yang lebih tinggi pada batas minimal dibandingkan dengan investor konservatif.

In [ ]:
# korelasi antar fitur
plt.figure(figsize=(10, 8))

kolom_numerik = [
    'monthly_income', 'monthly_expense_total', 'emergency_fund',
    'investment_amount', 'debt_to_income_ratio', 'score_emergency'
]

# Menghitung korelasi Pearson
matriks_korelasi = df_final[kolom_numerik].corr()

sns.heatmap(
    matriks_korelasi,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    vmin=-1, vmax=1
)

plt.title('Peta Korelasi (Heatmap) Indikator Finansial User', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

- korelasi 'score emergency' dengan 'emergency fund' : dengan korelasi pearson (0.83), artinya semakin besar tabungan dana darurat (emergency fund) seseorang, maka skor keamanannya akan ikut naik.
- korelasi 'score emergency' dengan 'monthly_expense_total' : dengan korelasi pearson (-.49), artinya semakin boros pengeluaran bulanan seseorang maka skor keamanannya akan turun. Dan juga membuktikan rumus rasio (Tabungan / (Pengeluaran x 6)) untuk membagi cluster berjalan sempurna.
- korelasi 'monthly_income' dengan 'invest_amount' (-0,05) : berati user yang bergaji besar tidak menjamin akan berinvestasi dengan jumlah yang besar.
- korelasi 'monthly_income' dengan 'monthly_expanse_total' (-0.00): artinya tidak ada pola user yang bergaji tinggi memilki pengeluaran perbulan yang boros.